In [1]:
import os
import gc
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from transformers import AutoTokenizer, AutoModel, AutoConfig
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


torch: 2.10.0+cu128
CUDA available: True


In [3]:
SEED = 14  
SMOTE_RANDOM_STATE=42

MODEL_NAME = "microsoft/codebert-base"
MAX_LENGTH = 512          
NUM_CLASSES = 6
DROPOUT = 0.30            


BATCH_SIZE = 8             
LEARNING_RATE = 1e-5       
WEIGHT_DECAY = 0.01        
FOCAL_GAMMA = 2.0          
MAX_EPOCHS = 30            
EARLY_STOP_PATIENCE = 10   
GRAD_CLIP_NORM = 1.0       


NUM_TEST_PROJECTS_PER_FOLD = 25   
MIN_TEST_EXAMPLES_PER_CLASS = 4   
VALIDATION_FRACTION = 0.20        


USE_SMOTE_OVERSAMPLING = True
SMOTE_TARGET_PER_FLAKY_CLASS = 500   

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


Device: cuda


In [4]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed()
print("Seed set to", SEED)


Seed set to 14


## Dataset



In [5]:
DATA_PATH = "/kaggle/input/datasets/tamim2500/flakylens-dataset/FlakyLens_dataset_with_nonflaky_indented.csv"  

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())


Shape: (8574, 6)


,id,project,test_name,full_code,label,category
0,1,apache_hadoop,TestDelegationTokenRenewer.testAddRemoveRenewA...,@Test\npublic void testAddRemoveRenewAction() ...,async wait,0
1,2,neo4j_neo4j,RobustJobSchedulerWrapperTest.shouldBeAbleToCa...,@Test\npublic void shouldBeAbleToCancelJob() t...,concurrency,1
2,3,eclipse_xtext-core,RequestManagerTest.testRunWriteAfterRead,@Test\npublic void testRunWriteAfterRead() {\n...,concurrency,1
3,4,NationalSecurityAgency_timely,MetricAdapterTest.testToMetricResponse,@Test\npublic void testToMetricResponse() thro...,unordered collections,3
4,5,apache_hadoop,TestPathData.testToFile,@Test\npublic void testToFile() throws Excepti...,test order dependency,4


In [7]:
LABEL_NAMES = {
    0: "Async Wait",
    1: "Concurrency",
    2: "Time",
    3: "Unordered Collections",
    4: "Test Order Dependency",
    5: "Non-flaky",
}

def normalize_dataset(df):
    df = df.copy()
    required = {"project", "full_code", "category"}
    if not required.issubset(df.columns):
        raise ValueError(f"Expected columns {required}, found {df.columns.tolist()}")

    df = df[["project", "full_code", "category"]].copy()
    df["project"] = df["project"].astype(str)
    df["full_code"] = df["full_code"].fillna("").astype(str)
    df["category"] = pd.to_numeric(df["category"], errors="raise").astype(int)

    if not set(df["category"].unique()).issubset(set(range(NUM_CLASSES))):
        raise ValueError("category values must be integers 0-5")

    return df.reset_index(drop=True)

data = normalize_dataset(df)
print("Number of projects:", data["project"].nunique())
print("\nClass distribution:")
display(data["category"].value_counts().sort_index().rename(index=LABEL_NAMES).to_frame("count"))
print("\nFlaky %:", round(100 * (data["category"] != 5).mean(), 2))


Number of projects: 98

Class distribution:


,count
category,
Async Wait,76
Concurrency,37
Time,33
Unordered Collections,41
Test Order Dependency,93
Non-flaky,8294



Flaky %: 3.27


In [8]:
def create_train_test_groups(df, num_test_projects=NUM_TEST_PROJECTS_PER_FOLD,
                              min_examples_per_class=MIN_TEST_EXAMPLES_PER_CLASS, seed=SEED):
    rng = random.Random(seed)
    unique_projects = df["project"].drop_duplicates().tolist()
    rng.shuffle(unique_projects)

    num_groups = math.ceil(len(unique_projects) / num_test_projects)
    groups = []
    used_projects = set()

    for _ in range(num_groups - 1):
        available_projects = [p for p in unique_projects if p not in used_projects]
        test_projects = available_projects[:num_test_projects]
        test_dataset = df[df["project"].isin(test_projects)]

        
        for _attempt in range(2000):
            category_counts = test_dataset["category"].value_counts()
            missing = [c for c in range(NUM_CLASSES) if category_counts.get(c, 0) < min_examples_per_class]
            if not missing:
                break
            rng.shuffle(available_projects)
            test_projects = available_projects[:num_test_projects]
            test_dataset = df[df["project"].isin(test_projects)]
        else:
            raise RuntimeError("Could not satisfy the minimum-per-class test requirement.")

        used_projects.update(test_projects)
        train_dataset = df[~df["project"].isin(test_projects)]
        groups.append((train_dataset.copy(), test_dataset.copy()))

  
    remaining = [p for p in unique_projects if p not in used_projects]
    if remaining:
        test_dataset = df[df["project"].isin(remaining)]
        train_dataset = df[~df["project"].isin(remaining)]
        groups.append((train_dataset.copy(), test_dataset.copy()))

    return groups

fold_groups = create_train_test_groups(data)
print(f"Created {len(fold_groups)} fold(s).\n")
for i, (train_pool, test_df) in enumerate(fold_groups, 1):
    overlap = set(train_pool["project"]) & set(test_df["project"])
    assert not overlap, f"Fold {i}: project leakage detected!"
    print(f"Fold {i}: {test_df['project'].nunique()} test projects | "
          f"train rows={len(train_pool)} | test rows={len(test_df)} | "
          f"test class counts={test_df['category'].value_counts().sort_index().to_dict()}")


Created 4 fold(s).

Fold 1: 25 test projects | train rows=6461 | test rows=2113 | test class counts={0: 26, 1: 8, 2: 10, 3: 6, 4: 9, 5: 2054}
Fold 2: 25 test projects | train rows=6411 | test rows=2163 | test class counts={0: 31, 1: 9, 2: 13, 3: 8, 4: 32, 5: 2070}
Fold 3: 25 test projects | train rows=6277 | test rows=2297 | test class counts={0: 11, 1: 10, 2: 6, 3: 15, 4: 43, 5: 2212}
Fold 4: 23 test projects | train rows=6573 | test rows=2001 | test class counts={0: 8, 1: 10, 2: 4, 3: 12, 4: 9, 5: 1958}


## CodeBERT architecture

In [44]:
class BERT_Arch(nn.Module):
    def __init__(self, auto_model, num_classes=NUM_CLASSES):
        super().__init__()
        self.bert = auto_model
        self.dropout = nn.Dropout(DROPOUT)
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(768, 512)
        self.fc2 = nn.Linear(512, num_classes)
        self.log_softmax = nn.LogSoftmax(dim=-1)

    def forward(self, input_ids, attention_mask):
        input_ids = input_ids.long()
        attention_mask = attention_mask.long()
        outputs = self.bert(input_ids, attention_mask=attention_mask)
        pooled = outputs[1] 

        x = self.fc1(pooled)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return self.log_softmax(x)


def build_fresh_model():

    model_config = AutoConfig.from_pretrained(MODEL_NAME, return_dict=False, output_hidden_states=True)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    auto_model = AutoModel.from_pretrained(MODEL_NAME, config=model_config)
    model = BERT_Arch(auto_model, NUM_CLASSES).to(DEVICE)
    return model, tokenizer


In [45]:
USE_RAW_LOGITS_LOSS = False  

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, model_output, targets):
  
        ce_loss = F.cross_entropy(model_output, targets, reduction="none", weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        if self.reduction == "mean":
            return focal_loss.mean()
        if self.reduction == "sum":
            return focal_loss.sum()
        return focal_loss


def compute_class_balanced_focal_loss(train_labels):
    class_weights_np = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(NUM_CLASSES),
        y=train_labels,
    )
    weights = torch.tensor(class_weights_np, dtype=torch.float32, device=DEVICE)
    return FocalLoss(alpha=weights, gamma=FOCAL_GAMMA), weights


In [46]:
def apply_smote_like_released_code(train_df):
 
    from imblearn.over_sampling import SMOTE
    from sklearn.feature_extraction.text import TfidfVectorizer

    x_text = train_df["full_code"].reset_index(drop=True)
    y = train_df["category"].reset_index(drop=True)

    vectorizer = TfidfVectorizer(max_features=5000)
    x_vec = vectorizer.fit_transform(x_text)

    sampling_strategy = {c: SMOTE_TARGET_PER_FLAKY_CLASS for c in range(5)}  
    counts = y.value_counts().to_dict()
    sampling_strategy = {c: n for c, n in sampling_strategy.items() if counts.get(c, 0) < n}

    if not sampling_strategy:
        return train_df  

    smote = SMOTE(sampling_strategy=sampling_strategy, random_state=SMOTE_RANDOM_STATE)
    x_res, y_res = smote.fit_resample(x_vec, y)

    num_synthetic = x_res.shape[0] - len(x_text)
    resampled_text = list(x_text) + ["[SMOTE-generated-sample]"] * num_synthetic

    return pd.DataFrame({"full_code": resampled_text, "category": y_res})


In [47]:
def tokenize_texts(tokenizer, texts):
    return tokenizer(
        texts.tolist(),
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )

def make_loader(tokens, labels, batch_size, shuffle):
    dataset = TensorDataset(tokens["input_ids"], tokens["attention_mask"],
                             torch.tensor(labels, dtype=torch.long))
    sampler = RandomSampler(dataset) if shuffle else SequentialSampler(dataset)
    return DataLoader(dataset, sampler=sampler, batch_size=batch_size)


## Train

In [48]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for input_ids, attention_mask, labels in loader:
        input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)  
        optimizer.step()
        total_loss += loss.item()
    return total_loss / max(1, len(loader))


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    preds_all, labels_all = [], []
    for input_ids, attention_mask, labels in loader:
        input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        total_loss += loss.item()
        preds_all.append(torch.argmax(outputs, dim=1).cpu().numpy())
        labels_all.append(labels.cpu().numpy())
    avg_loss = total_loss / max(1, len(loader))
    return avg_loss, np.concatenate(preds_all), np.concatenate(labels_all)


In [50]:
class EarlyStopping:
    def __init__(self, patience=EARLY_STOP_PATIENCE, delta=0.0):
        self.patience = patience
        self.delta = delta
        self.counter = 0
        self.best_score = None
        self.best_state = None
        self.early_stop = False

    def __call__(self, valid_f1, model):
        score = valid_f1
        if self.best_score is None or score > self.best_score + self.delta:
            self.best_score = score
            self.best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True


In [53]:
def train_one_fold(fold_idx, train_pool, test_df, output_dir):
    print(f"\n{'='*70}\nFOLD {fold_idx}\n{'='*70}")

  
    model, tokenizer = build_fresh_model()


    train_df, valid_df = train_test_split(
        train_pool, test_size=VALIDATION_FRACTION, random_state=SEED,
        stratify=train_pool["category"],
    )
    train_df = train_df.reset_index(drop=True)
    valid_df = valid_df.reset_index(drop=True)

    assert not (set(train_df["project"]) & set(test_df["project"]))
    assert not (set(valid_df["project"]) & set(test_df["project"]))

    if USE_SMOTE_OVERSAMPLING:
        train_df = apply_smote_like_released_code(train_df)

    print(f"train={len(train_df)}  valid={len(valid_df)}  test={len(test_df)}")

   
    tokens_train = tokenize_texts(tokenizer, train_df["full_code"])
    tokens_valid = tokenize_texts(tokenizer, valid_df["full_code"])
    tokens_test = tokenize_texts(tokenizer, test_df["full_code"])

    train_loader = make_loader(tokens_train, train_df["category"].to_numpy(), BATCH_SIZE, shuffle=True)
    valid_loader = make_loader(tokens_valid, valid_df["category"].to_numpy(), BATCH_SIZE, shuffle=False)
    test_loader  = make_loader(tokens_test,  test_df["category"].to_numpy(),  BATCH_SIZE, shuffle=False)

    
    criterion, class_weights = compute_class_balanced_focal_loss(train_df["category"].to_numpy())
    print("Class weights:", {LABEL_NAMES[i]: round(w, 3) for i, w in enumerate(class_weights.cpu().numpy())})

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    early_stopping = EarlyStopping(patience=EARLY_STOP_PATIENCE)

    history = []
    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        valid_loss, valid_preds, valid_labels = evaluate(model, valid_loader, criterion, DEVICE)
        valid_f1 = f1_score(valid_labels, valid_preds, average="macro", zero_division=0)

        history.append({"epoch": epoch, "train_loss": train_loss, "valid_loss": valid_loss, "valid_f1": valid_f1})
        print(f"  epoch {epoch:02d} | train_loss={train_loss:.4f} | valid_loss={valid_loss:.4f} | valid_macroF1={valid_f1:.4f}")

        early_stopping(valid_f1, model)
        if early_stopping.early_stop:
            print(f"  Early stopping at epoch {epoch} (best valid macro-F1={early_stopping.best_score:.4f})")
            break

  
    if early_stopping.best_state is not None:
        model.load_state_dict(early_stopping.best_state)

    
    test_loss, test_preds, test_labels = evaluate(model, test_loader, criterion, DEVICE)
    report = classification_report(
        test_labels, test_preds, labels=list(range(NUM_CLASSES)),
        target_names=[LABEL_NAMES[i] for i in range(NUM_CLASSES)],
        output_dict=True, zero_division=0,
    )
    print("\nFold", fold_idx, "test macro-F1:", report["macro avg"]["f1-score"])

    
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), output_dir / f"flakylens_fold{fold_idx}.pt")

    del model
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "fold": fold_idx,
        "history": pd.DataFrame(history),
        "report": report,
        "test_preds": test_preds,
        "test_labels": test_labels,
        "confusion_matrix": confusion_matrix(test_labels, test_preds, labels=list(range(NUM_CLASSES))),
    }


## Folds



## Fold 1

In [18]:
OUTPUT_DIR = "/kaggle/working/flakylens_models"

fold_results = []
for i, (train_pool, test_df) in enumerate(fold_groups[:1], 1):
    result = train_one_fold(i, train_pool, test_df, OUTPUT_DIR)
    fold_results.append(result)


FOLD 1


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

train=7491  valid=1293  test=2113
Class weights: {'Async Wait': np.float32(2.497), 'Concurrency': np.float32(2.497), 'Time': np.float32(2.497), 'Unordered Collections': np.float32(2.497), 'Test Order Dependency': np.float32(2.497), 'Non-flaky': np.float32(0.25)}
  epoch 01 | train_loss=1.3490 | valid_loss=0.1208 | valid_macroF1=0.2596
  epoch 02 | train_loss=1.3070 | valid_loss=0.1152 | valid_macroF1=0.3500
  epoch 03 | train_loss=1.2957 | valid_loss=0.1088 | valid_macroF1=0.3666
  epoch 04 | train_loss=1.2751 | valid_loss=0.0855 | valid_macroF1=0.5163
  epoch 05 | train_loss=1.2662 | valid_loss=0.0671 | valid_macroF1=0.6465
  epoch 06 | train_loss=1.2494 | valid_loss=0.0634 | valid_macroF1=0.5767
  epoch 07 | train_loss=1.2316 | valid_loss=0.0545 | valid_macroF1=0.6258
  epoch 08 | train_loss=1.2268 | valid_loss=0.0578 | valid_macroF1=0.7694
  epoch 09 | train_loss=1.2200 | valid_loss=0.0927 | valid_macroF1=0.5115
  epoch 10 | train_loss=1.2171 | valid_loss=0.0475 | valid_macroF1=0.72

In [19]:
def load_fold_model_from_checkpoint(checkpoint_path):
    model, tokenizer = build_fresh_model()
    state_dict = torch.load(checkpoint_path, map_location=DEVICE)
    missing, unexpected = model.load_state_dict(state_dict, strict=True)
    model.eval()
    return model, tokenizer


def evaluate_fold_from_checkpoint(fold_idx, checkpoint_path, train_df, test_df):
    print(f"\n{'='*70}\nFOLD {fold_idx} -- evaluating checkpoint (no training)\n{'='*70}")
    print("Checkpoint:", checkpoint_path)

    model, tokenizer = load_fold_model_from_checkpoint(checkpoint_path)

    tokens_test = tokenize_texts(tokenizer, test_df["full_code"])
    test_loader = make_loader(tokens_test, test_df["category"].to_numpy(), BATCH_SIZE, shuffle=False)

    criterion, _ = compute_class_balanced_focal_loss(train_df["category"].to_numpy())

    test_loss, test_preds, test_labels = evaluate(model, test_loader, criterion, DEVICE)
    report = classification_report(
        test_labels, test_preds, labels=list(range(NUM_CLASSES)),
        target_names=[LABEL_NAMES[i] for i in range(NUM_CLASSES)],
        output_dict=True, zero_division=0,
    )
    print("Fold", fold_idx, "test macro-F1:", report["macro avg"]["f1-score"])

    del model
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "fold": fold_idx,
        "history": None,
        "report": report,
        "test_preds": test_preds,
        "test_labels": test_labels,
        "confusion_matrix": confusion_matrix(test_labels, test_preds, labels=list(range(NUM_CLASSES))),
    }

In [20]:
import pickle

In [22]:
train_pool, test_df = fold_groups[0]
result = evaluate_fold_from_checkpoint(1,"/kaggle/working/flakylens_models/flakylens_fold1.pt", train_pool, test_df)



FOLD 1 -- evaluating checkpoint (no training)
Checkpoint: /kaggle/working/flakylens_models/flakylens_fold1.pt


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Fold 1 test macro-F1: 0.6723209515418236


In [23]:
from pathlib import Path
import pickle


Path("/kaggle/working/flakylens_models").mkdir(parents=True, exist_ok=True)

with open("/kaggle/working/flakylens_models/fold1_result.pkl", "wb") as f:
    pickle.dump(result, f)

print("Saved.")

Saved.


In [24]:
import os
print(os.listdir("/kaggle/working/flakylens_models"))

['flakylens_fold1.pt', 'fold1_result.pkl']


## Fold 2

In [15]:
FOLD_TO_RUN = 2   

import pickle

OUTPUT_DIR = "/kaggle/working/flakylens_models"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

train_pool, test_df = fold_groups[FOLD_TO_RUN - 1]
result = train_one_fold(FOLD_TO_RUN, train_pool, test_df, OUTPUT_DIR)


with open(Path(OUTPUT_DIR) / f"fold{FOLD_TO_RUN}_result.pkl", "wb") as f:
    pickle.dump(result, f)

print(f"Fold {FOLD_TO_RUN} done. Test macro-F1: {result['report']['macro avg']['f1-score']:.4f}")


FOLD 2


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

train=5089  valid=1273  test=2212
Class weights: {'Async Wait': np.float32(20.194), 'Concurrency': np.float32(38.553), 'Time': np.float32(36.877), 'Unordered Collections': np.float32(32.622), 'Test Order Dependency': np.float32(77.106), 'Non-flaky': np.float32(0.171)}
  epoch 01 | train_loss=1.5820 | valid_loss=1.5250 | valid_macroF1=0.2422
  epoch 02 | train_loss=1.4901 | valid_loss=1.4444 | valid_macroF1=0.2521
  epoch 03 | train_loss=1.3293 | valid_loss=1.2347 | valid_macroF1=0.3688
  epoch 04 | train_loss=1.1830 | valid_loss=1.0581 | valid_macroF1=0.5068
  epoch 05 | train_loss=0.8763 | valid_loss=1.2214 | valid_macroF1=0.6551
  epoch 06 | train_loss=0.5784 | valid_loss=1.1086 | valid_macroF1=0.6609
  epoch 07 | train_loss=0.3657 | valid_loss=0.9701 | valid_macroF1=0.8792
  epoch 08 | train_loss=0.1376 | valid_loss=1.0861 | valid_macroF1=0.7764
  epoch 09 | train_loss=0.0518 | valid_loss=0.9762 | valid_macroF1=0.8665
  epoch 10 | train_loss=0.0739 | valid_loss=1.3318 | valid_macroF

## Fold 3

In [25]:
FOLD_TO_RUN = 3   

import pickle

OUTPUT_DIR = "/kaggle/working/flakylens_models"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

train_pool, test_df = fold_groups[FOLD_TO_RUN - 1]
result = train_one_fold(FOLD_TO_RUN, train_pool, test_df, OUTPUT_DIR)




FOLD 3


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

train=7365  valid=1256  test=2297
Class weights: {'Async Wait': np.float32(2.455), 'Concurrency': np.float32(2.455), 'Time': np.float32(2.455), 'Unordered Collections': np.float32(2.455), 'Test Order Dependency': np.float32(2.455), 'Non-flaky': np.float32(0.252)}
  epoch 01 | train_loss=1.3400 | valid_loss=0.1156 | valid_macroF1=0.2347
  epoch 02 | train_loss=1.3039 | valid_loss=0.1100 | valid_macroF1=0.3089
  epoch 03 | train_loss=1.3014 | valid_loss=0.1011 | valid_macroF1=0.4227
  epoch 04 | train_loss=1.2896 | valid_loss=0.0867 | valid_macroF1=0.3710
  epoch 05 | train_loss=1.2715 | valid_loss=0.0755 | valid_macroF1=0.5193
  epoch 06 | train_loss=1.2565 | valid_loss=0.0664 | valid_macroF1=0.5009
  epoch 07 | train_loss=1.2415 | valid_loss=0.0609 | valid_macroF1=0.7242
  epoch 08 | train_loss=1.2277 | valid_loss=0.0656 | valid_macroF1=0.7354
  epoch 09 | train_loss=1.2244 | valid_loss=0.0802 | valid_macroF1=0.6930
  epoch 10 | train_loss=1.2197 | valid_loss=0.0736 | valid_macroF1=0.7

In [26]:
with open(Path(OUTPUT_DIR) / f"fold{FOLD_TO_RUN}_result.pkl", "wb") as f:
    pickle.dump(result, f)

print(f"Fold {FOLD_TO_RUN} done. Test macro-F1: {result['report']['macro avg']['f1-score']:.4f}")

Fold 3 done. Test macro-F1: 0.7011


## Fold 4

In [54]:
FOLD_TO_RUN = 4   

import pickle

OUTPUT_DIR = "/kaggle/working/flakylens_models"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

train_pool, test_df = fold_groups[FOLD_TO_RUN - 1]
result = train_one_fold(FOLD_TO_RUN, train_pool, test_df, OUTPUT_DIR)


FOLD 4


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

train=7569  valid=1315  test=2001
Class weights: {'Async Wait': np.float32(2.523), 'Concurrency': np.float32(2.523), 'Time': np.float32(2.523), 'Unordered Collections': np.float32(2.523), 'Test Order Dependency': np.float32(2.523), 'Non-flaky': np.float32(0.249)}
  epoch 01 | train_loss=1.3479 | valid_loss=0.1365 | valid_macroF1=0.2419
  epoch 02 | train_loss=1.3040 | valid_loss=0.1302 | valid_macroF1=0.2538
  epoch 03 | train_loss=1.2945 | valid_loss=0.1272 | valid_macroF1=0.2580
  epoch 04 | train_loss=1.2781 | valid_loss=0.0867 | valid_macroF1=0.4747
  epoch 05 | train_loss=1.2600 | valid_loss=0.0785 | valid_macroF1=0.5825
  epoch 06 | train_loss=1.2441 | valid_loss=0.0621 | valid_macroF1=0.8381
  epoch 07 | train_loss=1.2196 | valid_loss=0.0647 | valid_macroF1=0.7345
  epoch 08 | train_loss=1.2115 | valid_loss=0.0657 | valid_macroF1=0.8290
  epoch 09 | train_loss=1.2080 | valid_loss=0.0710 | valid_macroF1=0.7936
  epoch 10 | train_loss=1.2061 | valid_loss=0.0677 | valid_macroF1=0.7